In [50]:
#pip install --user langchain langchain-core langchain-community pypdf pymupdf sentence-transformers chromadb

In [63]:
import nbformat

# Load the notebook
with open("/content/RAG_pipeline.ipynb", "r") as f:
    nb = nbformat.read(f, as_version=4)

# Fix the metadata widgets issue
if "widgets" in nb.metadata:
    if "state" not in nb.metadata["widgets"]:
        nb.metadata["widgets"]["state"] = {}

# Save the fixed notebook
with open("/content/RAG_pipeline.ipynb", "w") as f:
    nbformat.write(nb, f)

print("✅ Fixed!")

✅ Fixed!


In [54]:
from google.colab import userdata
import os

# 🔐 Securely load API keys from Colab Secrets
os.environ["GROQ_API_KEY"]   = userdata.get("GROQ_API_KEY")
os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")

print("✅ All API keys loaded successfully!")

✅ All API keys loaded successfully!


In [2]:
from langchain_core.documents import Document

In [3]:
sample_doc = Document(
    page_content = "Hello World",
    metadata = {"source": "https://www.google.com"}
)

In [4]:
sample_doc

Document(metadata={'source': 'https://www.google.com'}, page_content='Hello World')

In [5]:
type(sample_doc)

langchain_core.documents.base.Document

In [51]:
#!pip install langchain langchain-community

In [7]:
# Text data
from langchain_community.document_loaders.text import TextLoader

loader = TextLoader("Data/python.txt", encoding="utf-8")

In [8]:
document = loader.load()

In [9]:
document

[Document(metadata={'source': 'Data/python.txt'}, page_content='\nPython is a high-level, interpreted programming language that has become one of the most popular and widely used languages in the world. Created by Guido van Rossum and first released in 1991, Python emphasizes simplicity and readability, making it easy for beginners to learn while remaining powerful for experienced developers. Its clean and concise syntax allows programmers to write fewer lines of code compared to many other languages, enhancing productivity and maintainability. Python supports multiple programming paradigms, including procedural, object-oriented, and functional programming, which makes it versatile for a wide range of applications.\nSome key features and benefits of Python include:\n* Ease of Learning: Simple syntax and readability make Python beginner-friendly.\n* Versatility: Suitable for web development, data analysis, artificial intelligence, machine learning, scientific computing, automation, and 

In [ ]:
# PDF data
#from langchain_community.document_loaders.pdf import PyPDFLoader

#loader = PyPDFLoader("../Data/research2.pdf")
#document = loader.load()

#print(document)

## Ingestion pipeline

# Documents

In [10]:
# Data = > Documents
import os
from langchain_community.document_loaders.pdf import PyPDFLoader


In [11]:
def load_all_pdfs():
    folder_path = "Data/pdfs"
    num_docs = 0
    all_docs = []

    for filename in os.listdir(folder_path):
        if filename.lower().endswith(".pdf"):
            # Complete file path
            pdf_path = os.path.join(folder_path, filename)

            loader = PyPDFLoader(pdf_path)
            doc = loader.load()

            all_docs.extend(doc)
            num_docs +=1
    print("total docs:",num_docs)
    print("total pages:",len(all_docs))
    return all_docs


In [12]:
all_pdf_documents = load_all_pdfs()

total docs: 2
total pages: 32


# Chunks

In [52]:
#pip install --user langchain_text_splitters

In [14]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

def split_doc(documents, chunk_size = 500, chunk_overlap = 50):

    text_spitter = RecursiveCharacterTextSplitter(
        chunk_size = chunk_size,
        chunk_overlap = chunk_overlap
    )

    chunked_docs = text_spitter.split_documents(documents)
    return chunked_docs

In [15]:
chunks = split_doc(all_pdf_documents)

In [16]:
len(chunks)

320

# Embedding

In [17]:
from sentence_transformers import SentenceTransformer

In [18]:
class EmbeddingManager:
    def __init__(self, model_name = "all-MiniLM-L6-v2"):
        self.model_name = model_name
        print("loading model....", self.model_name)
        self.model = SentenceTransformer(self.model_name)
        print("Embedding dimensions = ", self.model.get_sentence_embedding_dimension())

    def generate_embeddings(self, text):
        embeddings = self.model.encode(text, show_progress_bar = True)
        print("embeddings shape = ", embeddings.shape)
        return embeddings

In [19]:
embedding_manager = EmbeddingManager()

loading model.... all-MiniLM-L6-v2


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding dimensions =  384


###
Vector store

In [20]:
import chromadb
import uuid # create indexes

In [21]:
class VectorStoreManager:
    def __init__(self, persist_directory="data/vector_store", collection_name="pdf_documents"):
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.collection = None
        self.client = None

        self._initialize_store()

    def _initialize_store(self):
        os.makedirs(self.persist_directory, exist_ok=True)

        # create a client
        self.client = chromadb.PersistentClient(path=self.persist_directory)

        # create the collection
        self.collection = self.client.get_or_create_collection(
            name=self.collection_name,
            metadata={"description": "vector store collection for pdf embeddings in RAG"}
        )

        print("initialized the vector store with collection:", self.collection_name)
        print("docs in collection:", self.collection.count())

    def add_documents(self, documents, embeddings):
        if len(documents) != len(embeddings):
            raise ValueError("num of documents does not match num of embeddings")


        # store => ids, embedding, document, metadata
        ids = []
        all_metadata = []
        documents_content = []
        embeddings_list = []

        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
            doc_id = f"doc_{uuid.uuid4()}"
            ids.append(doc_id)

            metadata = dict(doc.metadata)
            metadata["doc_index"] = i
            metadata["content_length"] = len(doc.page_content)
            all_metadata.append(metadata)

            documents_content.append(doc.page_content)

            embeddings_list.append(embedding.tolist())

            self.collection.add(
                ids=ids,
                metadatas=all_metadata,
                documents=documents_content,
                embeddings=embeddings_list
            )

        print("total documents added in vector store=", len(documents_content))
        print("docs in collection= ", self.collection.count())













In [22]:
Vector_store = VectorStoreManager()

initialized the vector store with collection: pdf_documents
docs in collection: 0


In [23]:
# data => documents => chunks => embeddings => store in vector store

texts = [doc.page_content for doc in chunks]

embedding = embedding_manager.generate_embeddings(texts)

Vector_store.add_documents(chunks, embedding)

Batches:   0%|          | 0/10 [00:00<?, ?it/s]

embeddings shape =  (320, 384)
total documents added in vector store= 320
docs in collection=  320


## Retrieval pipeline

In [24]:
from sklearn.metrics.pairwise import cosine_similarity

In [25]:
class RAGRetriever:
    def __init__(self, embedding_manager, vector_store):
        self.embedding_manager = embedding_manager
        self.vector_store = vector_store


    def retrieve(self, query, top_k=5, score_threshold=0.0):
        # query => embedding
        query_embeddings = self.embedding_manager.generate_embeddings([query])[0]

        # semantic search
        results = self.vector_store.collection.query(
            query_embeddings=[query_embeddings.tolist()],
            n_results=top_k
        )

        # cosine similarity
        retrieved_docs=[]

        if results["documents"] and results["documents"][0]:
            ids = results["ids"][0]
            metadatas = results["metadatas"][0]
            documents = results["documents"][0]
            distances = results["distances"][0]

            for i, (doc_id, metadata, document, distance) in enumerate(zip(ids, metadatas, documents, distances)):
                similarity_score = 1 - distance

                if similarity_score >= score_threshold:
                    retrieved_docs.append({
                        "id": doc_id,
                        "document": document,
                        "metadata": metadata,
                        "distance": distance,
                        "similarity_score": similarity_score,
                        "rank" : i + 1
                    })

            print(f"retrieved {len(retrieved_docs)} documents")

        else:
            print("no documents found")

        return retrieved_docs

In [26]:
rag_retriever = RAGRetriever(embedding_manager, Vector_store)

In [1]:
#rag_retriever.retrieve("What is encoder decoder")

## Integrate with LLMs

## OpenAI-GPT

In [55]:
openai_api_key = os.environ.get("OPENAI_API_KEY")

In [30]:
#!pip install langchain-openai

In [56]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(
    openai_api_key = openai_api_key,
    model = "gpt-5.4",
    temperature = 0.1,
    max_tokens = 1024

)

In [36]:
# generate our retrieval-augmented output

def generate_output(query, retriever, llm, top_k = 3):
    results = retriever.retrieve(query, top_k)

    context = "\n".join(doc["document"] for doc in results) if results else ""

    if not context:
      print("we found no relevant content for the given query")

    # prompt = context + query
    prompt = f"""use given context to generate the answer for the query
              Context :{context}
              Query : {query}"""
    response = llm.invoke(prompt) # expecting a string as prompt (GPT)

    return response.content




In [39]:
#answer = generate_output("Whjat is RAG?", rag_retriever, llm)

### Groq

In [57]:
groq_api_key   = os.environ.get("GROQ_API_KEY")

In [43]:
#!pip install langchain-groq

In [58]:
from langchain_groq import ChatGroq

llm = ChatGroq(
    groq_api_key = groq_api_key,
    model = "qwen/qwen3-32b",
    temperature = 0.1,
    max_tokens = 1024

)

In [59]:
# generate our retrieval-augmented output

def generate_output(query, retriever, llm, top_k = 3):
    results = retriever.retrieve(query, top_k)

    context = "\n".join(doc["document"] for doc in results) if results else ""

    if not context:
      print("we found no relevant content for the given query")

    # prompt = context + query
    prompt = f"""use given context to generate the answer for the query
              Context :{context}
              Query : {query}"""
    response = llm.invoke([prompt.format(context = context, query = query)]) # expecting a list as prompt (groq)

    return response.content

In [60]:
answer = generate_output("Whjat is RAG?", rag_retriever, llm)

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

embeddings shape =  (1, 384)
retrieved 3 documents


In [61]:
print(answer)

<think>
Okay, the user is asking, "Whjat is RAG?" First, I notice there's a typo in the query—probably "What is RAG?" intended. So, I need to correct that mentally. Now, looking at the provided context, the paper is a survey on RAG methods. The context mentions that RAG stands for Retrieval-Augmented Generation. 

The context explains that RAG combines retrieval from external sources with a generative model. The example given is ChatGPT using RAG to access recent news, which it can't do with just pre-training data. So, the main idea is that RAG enhances the model's knowledge by retrieving relevant information from external databases or the web when answering a query.

I should structure the answer to first define RAG, mention its components (retrieval and generation), and perhaps give an example like the ChatGPT scenario. Also, the context talks about different paradigms of RAG, like naive RAG and effective frameworks. Maybe mention that it's a systematic review covering state-of-the-a